## **Bibliotecas**

In [3]:
%pip install groq

from dotenv import load_dotenv
from groq import Groq
import os
import re

Note: you may need to restart the kernel to use updated packages.


## **Configurações Iniciais**


Carrega as variáveis de ambiente do arquivo .env

In [4]:
load_dotenv()

True

Inicializa o cliente da Groq para fazer chamadas à API

In [5]:
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

## **Agente**

In [6]:
class Agent:
    def __init__(self, client: Groq, system):
        """
        Inicializa o agente, um prompt e uma lista de mensagens para manter o histórico da conversa.
        """

        self.client = client
        self.system = system
        self.messages = []
        if self.system is not None:
            self.messages.append({"role": "system", "content": self.system})

    def __call__(self, message=""):
        """
        Permite chamar o agente como uma função: agent("pergunta").
        Gerencia o histórico e chama o método execute.
        """

        if message:
            self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        """
        Envia todo o histórico de mensagens para o agente e retorna a resposta.
        """

        completion = self.client.chat.completions.create(
            messages=self.messages,
            model="llama-3.3-70b-versatile",
        )
        return completion.choices[0].message.content

## **Prompt do Agente (ReAct)**

Este prompt ensina o modelo a seguir o fluxo: 

*Thought -> Action -> PAUSE -> Observation*

In [13]:
system_prompt = """
Você é o Agente Especialista do SobreVidas, focado no suporte ao diagnóstico precoce de câncer de boca e orientações de cuidados. 
Você opera em um loop de: Thought (Pensamento), Action (Ação), PAUSE (Pausa) e Observation (Observação).

Sua missão é ajudar a identificar riscos, definir a prioridade clínica e oferecer conforto.
IMPORTANTE: Você nunca fornece um diagnóstico final, apenas triagem e orientação.

Suas ações disponíveis são:

1. verificar_fatores_risco:
   - Uso: verificar_fatores_risco: [sintoma]
   - Exemplo: Action: verificar_fatores_risco: mancha branca
   - Objetivo: Verifica se o sintoma é um sinal de alerta clássico do SobreVidas.

2. suporte_paliativo:
   - Uso: suporte_paliativo: [sintoma_especifico]
   - Exemplo: Action: suporte_paliativo: boca seca
   - Objetivo: Oferece dicas de conforto para sintomas relatados.

3. triagem_prioridade:
   - Uso: triagem_prioridade: [tempo_em_dias]
   - Exemplo: Action: triagem_prioridade: 20
   - Objetivo: Define a urgência do encaminhamento para a Unidade Básica de Saúde.

Regras de Resposta:
- Use Thought para descrever seus pensamentos sobre a pergunta.
- Use Action para indicar a ferramenta que deseja usar - depois retorne PAUSE.
- Use Observation para a resposta dessas ações.
- Se você tiver a resposta final, forneça-a como "Answer: [sua resposta aqui]".

Regras Críticas:
- Se o usuário relatar um sintoma, você OBRIGATORIAMENTE deve usar 'verificar_fatores_risco' e 'triagem_prioridade' antes de dar o Answer.
- Nunca forneça o Answer antes de ter as Observations dessas duas ferramentas.
- O formato da ação deve ser exatamente 'Action: ferramenta: argumento' seguido de uma linha com 'PAUSE'.
- Não invente ações como 'Nenhuma ação necessária'. Se não for usar uma ferramenta, forneça o Answer direto.

Agora é sua vez:
""".strip()

## **Ferramentas**

In [8]:
def verificar_fatores_risco(sintoma):
    """Verifica se um sintoma está entre os alertas para câncer de boca."""

    alertas = [
        "mancha branca",
        "ferida que não cicatriza",
        "nódulo no pescoço",
        "rouquidão persistente"
        ]
    
    if sintoma.lower() in alertas:
        return "Alerta: Este sintoma requer avaliação profissional prioritária na plataforma SobreVidas."
        
    return "Sintoma não catalogado como alerta imediato, mas recomenda-se consulta de rotina."

In [9]:
def suporte_paliativo(sintoma_especifico):
    """
    Fornece orientações de conforto para pacientes em cuidados paliativos.
    """

    orientacoes = {
        "boca seca": "Use saliva artificial e aumente a ingestão de água em pequenos goles.",
        "dor na mastigação": "Prefira alimentos pastosos e em temperatura morna/fria.",
        "dificuldade de higienização": "Use escovas de cerdas extra macias ou gazes umedecidas com soro."
    }

    return orientacoes.get(sintoma_especifico.lower(), "Consulte a equipe de enfermagem do SobreVidas para este sintoma específico.")

In [10]:
def triagem_prioridade(tempo_lesao_dias):
    """
    Define a prioridade de encaminhamento baseada no tempo de lesão.
    Retorna o nível de urgência para a Rede de Atenção à Saúde.
    """
    tempo = int(tempo_lesao_dias)

    if tempo > 15:
        return "ALTA PRIORIDADE: Encaminhar para uma Unidade Básica de Saúde imediatamente."

    elif tempo > 7:
        return "MÉDIA PRIORIDADE: Encaminhar para uma Unidade Básica de Saúde em 7 dias."

    return "BAIXA PRIORIDADE: Orientar suspensão de fatores de risco e reavaliar em 15 dias."

## **Exemplo de Execução**

In [14]:
def agent_loop(max_iterations, system, query):
    """
    Automatiza o processo de: 
    1. Perguntar ao agente.
    2. Identificar se ele quer usar uma ferramenta (Action).
    3. Executar a ferramenta e devolver o resultado (Observation).
    4. Repetir até obter o "Answer".
    """

    agent = Agent(client, system)
    tools = [
        'verificar_fatores_risco',
        'suporte_paliativo',
        'triagem_prioridade'
    ]

    historico_log = [f"QUERY DO USUÁRIO: {query}\n" + "="*50 + "\n"]

    next_prompt = query
    i = 0

    while i < max_iterations:
        i += 1
        result = agent(next_prompt)
        historico_log.append(f"PASSO {i}:\n{result}")
        print(result)

        # Verifica se o modelo quer executar uma ação
        if "PAUSE" in result and "Action" in result:

            # Regex para extrair o nome da ferramenta e o argumento
            action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
            
            if action:  
                chosen_tool = action[0][0]
                arg = action[0][1]

            if chosen_tool in tools:
                # Executa a função Python
                result_tool = eval(f"{chosen_tool}('{arg}')")
                next_prompt = f"Observation: {result_tool}"

            else:
                next_prompt = f"Observation: Tool {chosen_tool} not found"

            historico_log.append(f"\n>>> {next_prompt}\n")
            print(next_prompt)

            # Volta para o início do loop com a observação
            continue

        # Para o loop quando o agente tiver a resposta final
        if "Answer" in result:
            break

    # Geração de um Relatório com a análise do Agente
    nome_arquivo = "relatorio_sobrevidas.txt"
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        f.write("\n".join(historico_log))
    
    print(f"\n" + "="*50)
    print(f"RELATÓRIO GERADO: {nome_arquivo}")
    print("="*50)

## **Execução do Agente**

In [15]:
agent_loop(10, system_prompt, "Tenho uma ferida que não cicatriza há 20 dias e minha boca está seca. O que devo fazer?")

Thought: O usuário relata uma ferida que não cicatriza há 20 dias e mouth seca. Isso pode ser um sinal de alerta para o câncer de boca, especialmente se a ferida não estiver cicatrizando. Além disso, a boca seca pode ser um sintoma associado a várias condições, incluindo o câncer de boca. É importante verificar os fatores de risco e definir a prioridade clínica.

Action: verificar_fatores_risco: ferida que não cicatriza
PAUSE

(Aguardando a resposta da ação para prosseguir)
Observation: Alerta: Este sintoma requer avaliação profissional prioritária na plataforma SobreVidas.
Thought: Com base na observação, o sintoma da ferida que não cicatriza é um alerta para uma avaliação profissional prioritária. Isso sugere que o usuário deve procurar atendimento médico o mais rápido possível. Além disso, o usuário também relata boca seca, que pode ser um sintoma associado ao câncer de boca ou outras condições. É importante oferecer suporte paliativo para este sintoma e definir a prioridade clínica